# 선택된 패치만으로 영상 복원하기

AutoGaze가 선택한 일부 패치만으로 **원본 영상 전체를 복원**하는 과정을 재현합니다.

**다루는 내용**
1. 복원 원리 — Masked Autoencoder (MAE) 와 AutoGaze의 관계
2. VideoMAE 모델 로드
3. AutoGaze로 패치 선택 → VideoMAE로 전체 프레임 복원
4. 시각화: 원본 / 선택 패치 / 복원 결과 비교
5. `gazing_ratio`에 따른 복원 품질 변화
6. 정량 평가 (PSNR · SSIM)
7. 복원 영상 저장

**사전 조건**
```bash
source .venv/bin/activate
# VideoMAE 가중치 필요 (~2 GB)
bash scripts/download_models.sh
```

---
## 1. 복원 원리

### Masked Autoencoder (MAE) 기초

MAE (He et al., 2021, "Masked Autoencoders Are Scalable Vision Learners") 의 핵심 아이디어:

```
전체 패치 중 일부(visible)만 ViT Encoder에 통과
  ↓
Encoder 출력(visible features) + 학습 가능한 mask_token (missing positions)
  ↓
Decoder → 마스크된 위치의 픽셀 값 예측
  ↓
원본과 비교 → reconstruction loss 최소화로 학습
```

**VideoMAE** 는 이를 비디오로 확장: 프레임 단위 + 시간 축 causal masking.

### AutoGaze와의 연결고리

AutoGaze는 VideoMAE reconstruction을 **학습 신호**로 사용합니다:

```
AutoGaze 선택 패치 → VideoMAE 인코더 → VideoMAE 디코더 → 복원 프레임
                                                          ↓
                                               reconstruction loss
                                                          ↓
                                    AutoGaze RL reward = -loss
```

> **결론**: AutoGaze는 *VideoMAE가 전체 영상을 잘 복원할 수 있는 최소 패치 집합*을 찾도록 학습됩니다.  
> 따라서 선택된 패치 + VideoMAE 디코더 = 전체 영상 복원 가능.

### 복원 파이프라인 (코드 기준)

```
video (B, T, C, H, W)
    │
    ├─ AutoGaze ──→ gazing_pos, gazing_mask   ← 선택 패치 인덱스
    │
    └─ VideoMAE.vit (Encoder)
           입력: visible patches only (gazing_pos 위치)
           출력: latent (B, N_visible, D)
               │
           VideoMAE.decoder
               입력: latent + mask_token @ ALL positions
               출력: logits (B, T, N_patches, patch_size²×C)
               │
           unpatchify → reconstruction (B, T, C, 224, 224)
```

### 복원 품질 한계

| gazing_ratio | 선택 패치 수 | 복원 품질 |
| --- | --- | --- |
| 0.10 | ~26개/프레임 | 윤곽·색상은 복원, 세부 텍스처 손실 |
| 0.25 | ~66개/프레임 | 주요 객체 복원 |
| 0.50 | ~132개/프레임 | 사실상 원본에 가까움 |
| 0.75 | ~198개/프레임 | 원본과 거의 동일 |

전체 패치 수: 265개/프레임 (32²+64²+112²+224² → 16px 패치 기준 합산)

---
## 0. 환경 확인

In [ ]:
import sys, platform
import matplotlib
import matplotlib.font_manager as fm
import torch

def _setup_korean_font():
    _sys = platform.system()
    if _sys == 'Darwin':
        matplotlib.rcParams['font.family'] = 'AppleGothic'
        _font = 'AppleGothic'
    elif _sys == 'Windows':
        matplotlib.rcParams['font.family'] = 'Malgun Gothic'
        _font = 'Malgun Gothic'
    else:
        # NanumGothic 우선 (sudo apt-get install fonts-nanum && fc-cache -fv)
        _available = {f.name for f in fm.fontManager.ttflist}
        _preferred = ['NanumGothic', 'NanumBarunGothic', 'NanumGothicCoding', 'NanumMyeongjo']
        _candidates = [f for f in _preferred if f in _available]
        if not _candidates:
            _candidates = [f.name for f in fm.fontManager.ttflist
                           if any(k in f.name for k in ('Nanum', 'UnDotum', 'Baekmuk', 'Gothic'))]
        if _candidates:
            matplotlib.rcParams['font.family'] = _candidates[0]
            _font = _candidates[0]
        else:
            print("⚠  한글 폰트 없음 — sudo apt-get install fonts-nanum && fc-cache -fv")
            return
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"[폰트] {_font}")

_setup_korean_font()

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Device : CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Device : MPS (Apple Silicon)")
else:
    print("Device : CPU")
import autogaze; print("autogaze ✓")
from autogaze.utils import patch_transformers_for_torch25; patch_transformers_for_torch25()

In [ ]:
from pathlib import Path

# ── 경로 설정 ────────────────────────────────────────────────────
AUTOGAZE_PATH = "../weights/AutoGaze"          # 로컬 또는 "nvidia/AutoGaze"
VIDEOMAE_PT   = "../weights/VideoMAE_AutoGaze/videomae.pt"
VIDEO_PATH    = "../assets/example_input.mp4"
OUTPUT_DIR    = Path("../results/reconstruction")
# ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

videomae_available = Path(VIDEOMAE_PT).exists()
if not videomae_available:
    print(f"⚠  VideoMAE 가중치 없음: {VIDEOMAE_PT}")
    print("   bash scripts/download_models.sh 로 다운로드하세요.")
else:
    print(f"VideoMAE 가중치: {VIDEOMAE_PT}  ({Path(VIDEOMAE_PT).stat().st_size/1e9:.2f} GB)")

assert Path(VIDEO_PATH).exists(), f"비디오 없음: {VIDEO_PATH}"
print(f"비디오: {VIDEO_PATH}")

---
## 2. 모델 로드 — AutoGaze + VideoMAE

In [ ]:
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor
from autogaze.utils import get_device

device = get_device()
print(f"디바이스: {device}")

# AutoGaze
print("\nAutoGaze 로드 중...")
ag_transform = AutoGazeImageProcessor.from_pretrained(AUTOGAZE_PATH)
ag_model     = AutoGaze.from_pretrained(AUTOGAZE_PATH).to(device)
ag_model.eval()

CHUNK_SIZE = ag_model.config.max_num_frames       # 16
NUM_TOKENS = ag_model.config.num_vision_tokens_each_frame  # 265
SCALES     = [int(s) for s in ag_model.config.scales.split("+")]  # [32,64,112,224]
print(f"AutoGaze 로드 완료 ✓  (frames={CHUNK_SIZE}, tokens/frame={NUM_TOKENS}, scales={SCALES})")

In [ ]:
if not videomae_available:
    print("VideoMAE 가중치 없음 — 이 셀을 건너뜁니다.")
    mae_model = None
else:
    from omegaconf import OmegaConf
    from autogaze.tasks.video_mae_reconstruction.modeling_video_mae import ViTMAEForPreTraining
    from transformers import VivitImageProcessor

    # VideoMAE 설정 — weights/VideoMAE_AutoGaze/config.yaml 에서 확인한 값
    # l1만 사용 (dinov2_reg, siglip2는 손실 계산용으로 복원에는 불필요)
    VIDEOMAE_CONFIG = OmegaConf.create({
        "loss_type":       "l1",
        "loss_weights":    "1",
        "scale_embed":     True,
        "time_embed":      True,
        "causal":          True,
        "max_num_frames":  256,
    })
    VIDEOMAE_BASE   = "facebook/vit-mae-large"
    VIDEOMAE_SCALES = "32+64+112+224"

    print("VideoMAE 아키텍처 초기화 중 (facebook/vit-mae-large) ...")
    mae_model = ViTMAEForPreTraining.from_pretrained(
        VIDEOMAE_BASE,
        attn_implementation="sdpa",
        scales=VIDEOMAE_SCALES,
        **OmegaConf.to_container(VIDEOMAE_CONFIG),
    )

    # 커스텀 학습 가중치 로드
    print("VideoMAE 커스텀 가중치 로드 중 ...")
    state = torch.load(VIDEOMAE_PT, map_location="cpu", weights_only=True)
    missing, unexpected = mae_model.load_state_dict(state, strict=False)
    print(f"  missing keys  : {len(missing)}  (dinov2_reg, siglip2 관련 — 정상)")
    print(f"  unexpected keys: {len(unexpected)}")

    mae_model = mae_model.to(device).eval()
    mae_transform = VivitImageProcessor.from_pretrained(VIDEOMAE_BASE, size=224)
    print("VideoMAE 로드 완료 ✓")

---
## 3. 비디오 로드 및 전처리

In [ ]:
import av
import numpy as np
from autogaze.datasets.video_utils import (
    read_video_pyav, sample_frame_indices, process_video_frames,
    transform_video_for_pytorch,
)
from autogaze.utils import UnNormalize

video_path = Path(VIDEO_PATH)

# 비디오 메타데이터
container = av.open(str(video_path))
stream = container.streams.video[0]
total_frames = stream.frames
fps = float(stream.average_rate)
container.close()
print(f"비디오: {video_path.name}  ({total_frames}프레임, {fps:.1f}fps)")

# 16프레임 샘플링
container = av.open(str(video_path))
indices   = sample_frame_indices(
    clip_len=CHUNK_SIZE, frame_sample_rate=1,
    seg_len=total_frames, random_sample_frame=False,
)
raw_video = read_video_pyav(container, indices)   # (T, H, W, 3) uint8
container.close()
raw_video = process_video_frames(raw_video, CHUNK_SIZE)
T = raw_video.shape[0]
print(f"로드된 프레임: {T}개  shape={raw_video.shape}")

# AutoGaze 입력 텐서 (224×224 리사이즈)
video_ag   = transform_video_for_pytorch(raw_video, ag_transform)
video_ag_b = video_ag[None].to(device)  # (1, T, C, H, W)

# VideoMAE 입력 텐서 (동일 해상도 사용 가능)
if mae_model is not None:
    video_mae   = transform_video_for_pytorch(raw_video, mae_transform)
    video_mae_b = video_mae[None].to(device)  # (1, T, C, H, W)

# 시각화용 정규화 해제 함수
unnorm_ag = UnNormalize(
    ag_transform.image_mean, ag_transform.image_std,
    getattr(ag_transform, 'rescale_factor', 1/255.0),
)
video_vis = unnorm_ag(video_ag).cpu().float().numpy()  # (T, C, H, W) in [0,1]

print("전처리 완료 ✓")

---
## 4. AutoGaze → VideoMAE 복원 파이프라인

In [ ]:
from autogaze.datasets.collate import process_gazing_info
import json

def autogaze_to_mae_gazing_info(gaze_outputs, device):
    """
    AutoGaze 출력(gaze_outputs)을 VideoMAE decoder가 받는
    gazing_info 형식으로 변환합니다.
    
    AutoGaze의 gazing_pos 는 trainer의 _one_step_ntp 에서 사용하는
    process_gazing_info 와 동일한 구조를 이미 가지고 있습니다.
    """
    pos          = gaze_outputs["gazing_pos"]          # (1, N)
    if_padded    = gaze_outputs["if_padded_gazing"]    # (1, N) bool
    num_each     = gaze_outputs["num_gazing_each_frame"]  # (T,)
    num_tokens   = gaze_outputs["num_vision_tokens_each_frame"]

    gazing_info = {
        "gazing_pos":             pos.to(device),
        "if_padded_gazing":       if_padded.to(device),
        "num_gazing_each_frame":  num_each.to(device),
        "original_seq_length":    torch.tensor(T * num_tokens, device=device),
    }
    return gazing_info


def reconstruct(video_ag_b, video_mae_b, gazing_ratio=0.75, task_loss_req=0.7):
    """
    AutoGaze로 패치 선택 후 VideoMAE로 전체 프레임 복원.
    
    Returns:
        gaze_outputs : AutoGaze 원본 출력
        recon_frames : (T, C, H, W) float32 [0,1] — 복원된 프레임 (unnormalized)
        recon_loss   : scalar — 평균 복원 손실
    """
    # 1. AutoGaze — 어떤 패치를 볼지 결정
    with torch.inference_mode():
        gaze_outputs = ag_model(
            {"video": video_ag_b},
            gazing_ratio=gazing_ratio,
            task_loss_requirement=task_loss_req,
        )

    if mae_model is None:
        return gaze_outputs, None, None

    # 2. gazing_info 변환
    gazing_info = autogaze_to_mae_gazing_info(gaze_outputs, device)

    # 3. VideoMAE — 선택된 패치로 전체 프레임 복원
    frame_idx_all = torch.arange(T, device=device)  # 모든 프레임 복원

    with torch.inference_mode():
        mae_output = mae_model(
            video_mae_b,
            gazing_info=gazing_info,
            frame_idx_to_reconstruct=frame_idx_all,
            interpolate_pos_encoding=True,
        )

    # 4. 복원 결과 정규화 해제 (VideoMAE 정규화 기준)
    recon = mae_output.reconstruction[0]  # (T, C, H, W)
    unnorm_mae = UnNormalize(
        mae_transform.image_mean, mae_transform.image_std,
        getattr(mae_transform, 'rescale_factor', 1/255.0),
    )
    recon_vis = unnorm_mae(recon).cpu().float().clamp(0, 1).numpy()  # (T, C, H, W)

    loss = float(mae_output.loss_mean.mean().item())
    return gaze_outputs, recon_vis, loss


print("reconstruct() 함수 준비 완료 ✓")

In [ ]:
# ── 기본 실행: gazing_ratio=0.75, task_loss_requirement=0.7 ──────
GAZING_RATIO  = 0.75
TASK_LOSS_REQ = 0.7
# ─────────────────────────────────────────────────────────────────

print(f"gazing_ratio={GAZING_RATIO}, task_loss_requirement={TASK_LOSS_REQ}")
print("AutoGaze + VideoMAE 실행 중 ...")

gaze_outputs, recon_frames, recon_loss = reconstruct(
    video_ag_b, video_mae_b if mae_model else None,
    gazing_ratio=GAZING_RATIO,
    task_loss_req=TASK_LOSS_REQ,
)

n_real = int((~gaze_outputs["if_padded_gazing"]).sum().item())
print(f"\n선택된 패치: {n_real} / {T * NUM_TOKENS}  ({100*n_real/(T*NUM_TOKENS):.1f}%)")
if recon_loss is not None:
    print(f"복원 손실 (L1): {recon_loss:.4f}")

---
## 5. 시각화: 원본 / 선택 패치 / 복원 비교

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as F

def make_masked_view(frame_chw, gazing_mask_t, scale=224, dim=0.15):
    """선택된 패치는 밝게, 나머지는 어둡게 표시한 프레임 반환 (HWC uint8)."""
    ft = torch.from_numpy(frame_chw).unsqueeze(0)
    fs = F.interpolate(ft, size=(scale, scale), mode="bicubic", align_corners=False).squeeze()

    # 가장 큰 스케일 마스크를 224로 업샘플
    pg = int(gazing_mask_t.shape[-1] ** 0.5)
    m = F.interpolate(
        gazing_mask_t.reshape(pg, pg).unsqueeze(0).unsqueeze(0).float(),
        size=(scale, scale), mode="nearest",
    ).squeeze().numpy()

    masked = fs.numpy() * (dim + (1 - dim) * m[None])
    return (np.clip(masked.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)


def draw_patch_borders(ax, gazing_mask_t, scale=224):
    """선택된 패치에 빨간 테두리 그리기."""
    pg = int(gazing_mask_t.shape[-1] ** 0.5)
    patch_px = scale // pg
    mask_2d = gazing_mask_t.reshape(pg, pg).numpy()
    for pi in range(pg):
        for pj in range(pg):
            if mask_2d[pi, pj] > 0.5:
                ax.add_patch(mpatches.Rectangle(
                    (pj * patch_px - 0.5, pi * patch_px - 0.5),
                    patch_px, patch_px,
                    linewidth=0.8, edgecolor="red", facecolor="none",
                ))


# 시각화 대상 프레임 결정
SHOW_FRAMES = min(T, 8)  # 최대 8개
frame_indices = np.linspace(0, T - 1, SHOW_FRAMES, dtype=int)
largest_mask = gaze_outputs["gazing_mask"][-1][0]  # (T, N_large)

n_rows = 2 if mae_model is None else 3
row_labels = ["원본", "선택 패치 (AutoGaze)"]
if mae_model is not None:
    row_labels.append(f"VideoMAE 복원 (L1={recon_loss:.4f})")

fig, axes = plt.subplots(n_rows, SHOW_FRAMES, figsize=(SHOW_FRAMES * 2.5, n_rows * 2.5))
if SHOW_FRAMES == 1:
    axes = axes.reshape(n_rows, 1)

for col, t in enumerate(frame_indices):
    # Row 0: 원본
    orig_hw = (video_vis[t].transpose(1, 2, 0) * 255).astype(np.uint8)
    axes[0, col].imshow(orig_hw)
    axes[0, col].set_title(f"F{t+1}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: 선택 패치 (어두운 배경 + 빨간 테두리)
    masked_img = make_masked_view(video_vis[t], largest_mask[t].cpu())
    axes[1, col].imshow(masked_img)
    draw_patch_borders(axes[1, col], largest_mask[t].cpu())
    n_sel = int(largest_mask[t].sum().item())
    axes[1, col].set_title(f"Scale-224: {n_sel}패치", fontsize=7)
    axes[1, col].axis("off")

    # Row 2: VideoMAE 복원
    if mae_model is not None and recon_frames is not None:
        recon_hw = (recon_frames[t].transpose(1, 2, 0) * 255).astype(np.uint8)
        axes[2, col].imshow(recon_hw)
        axes[2, col].axis("off")

# 행 레이블
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=90, va="center")

plt.suptitle(
    f"원본 / 선택 패치 / VideoMAE 복원\n"
    f"(gazing_ratio={GAZING_RATIO}, task_loss_req={TASK_LOSS_REQ}, "
    f"선택 {n_real}/{T*NUM_TOKENS} = {100*n_real/(T*NUM_TOKENS):.0f}%)",
    fontsize=11,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "reconstruction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/reconstruction_comparison.png")

---
## 6. gazing_ratio에 따른 복원 품질 변화

In [ ]:
# 복원 품질 비교: 여러 gazing_ratio
RATIOS_TO_TEST = [0.1, 0.25, 0.5, 0.75]

ratio_results = {}
for ratio in RATIOS_TO_TEST:
    go, rf, rl = reconstruct(
        video_ag_b,
        video_mae_b if mae_model else None,
        gazing_ratio=ratio,
        task_loss_req=None,   # 임계값 없이 비율만 제어
    )
    n_sel = int((~go["if_padded_gazing"]).sum().item())
    ratio_results[ratio] = {
        "gaze_outputs": go,
        "recon_frames": rf,
        "loss": rl,
        "n_selected": n_sel,
    }
    loss_str = f", L1={rl:.4f}" if rl is not None else ""
    print(f"  ratio={ratio:.2f}: {n_sel}/{T*NUM_TOKENS}패치 ({100*n_sel/(T*NUM_TOKENS):.0f}%){loss_str}")

In [ ]:
# ── 참조 프레임 선택 (중간 프레임)
REF_FRAME = T // 2
lm_idx = len(SCALES) - 1  # 가장 큰 스케일

n_cols = len(RATIOS_TO_TEST)
n_rows = 2 if mae_model is None else 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
if n_cols == 1:
    axes = axes.reshape(n_rows, 1)

for col, ratio in enumerate(RATIOS_TO_TEST):
    r = ratio_results[ratio]
    go = r["gaze_outputs"]
    lm = go["gazing_mask"][lm_idx][0]  # (T, N)

    # Row 0: 선택 패치 시각화
    masked_img = make_masked_view(video_vis[REF_FRAME], lm[REF_FRAME].cpu())
    axes[0, col].imshow(masked_img)
    draw_patch_borders(axes[0, col], lm[REF_FRAME].cpu())
    axes[0, col].set_title(
        f"ratio={ratio}\n{r['n_selected']}/{T*NUM_TOKENS}패치",
        fontsize=9,
    )
    axes[0, col].axis("off")

    # Row 1: VideoMAE 복원
    if mae_model is not None and r["recon_frames"] is not None:
        recon_hw = (r["recon_frames"][REF_FRAME].transpose(1, 2, 0) * 255).astype(np.uint8)
        axes[1, col].imshow(recon_hw)
        axes[1, col].set_title(f"L1={r['loss']:.4f}", fontsize=9)
        axes[1, col].axis("off")

# 원본 행 추가 (비교 기준)
if mae_model is not None:
    for col in range(n_cols):
        orig_hw = (video_vis[REF_FRAME].transpose(1, 2, 0) * 255).astype(np.uint8)
        axes[2, col].imshow(orig_hw)
        if col == 0:
            axes[2, col].set_ylabel("원본", fontsize=10, rotation=0, labelpad=50, va="center")
        axes[2, col].axis("off")

axes[0, 0].set_ylabel("선택 패치", fontsize=10, rotation=0, labelpad=60, va="center")
if mae_model is not None:
    axes[1, 0].set_ylabel("VideoMAE 복원", fontsize=10, rotation=0, labelpad=75, va="center")

plt.suptitle(f"gazing_ratio별 복원 품질 비교  (Frame {REF_FRAME+1})", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ratio_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/ratio_comparison.png")

---
## 7. 정량 평가 — PSNR · SSIM

In [ ]:
if mae_model is None:
    print("VideoMAE 없음 — 이 섹션을 건너뜁니다.")
else:
    from skimage.metrics import peak_signal_noise_ratio as psnr
    from skimage.metrics import structural_similarity as ssim

    # VideoMAE 정규화 기준 원본 프레임
    unnorm_mae = UnNormalize(
        mae_transform.image_mean, mae_transform.image_std,
        getattr(mae_transform, 'rescale_factor', 1/255.0),
    )
    orig_mae_vis = unnorm_mae(video_mae).cpu().float().clamp(0, 1).numpy()  # (T, C, H, W)

    print(f"{'ratio':>8} | {'선택 패치':>10} | {'PSNR (dB)':>10} | {'SSIM':>8} | {'L1 loss':>10}")
    print("-" * 58)

    psnr_vals, ssim_vals, l1_vals = [], [], []

    for ratio in RATIOS_TO_TEST:
        r = ratio_results[ratio]
        if r["recon_frames"] is None:
            continue

        psnr_per_frame, ssim_per_frame = [], []
        for t in range(T):
            orig_t  = orig_mae_vis[t].transpose(1, 2, 0)        # HWC
            recon_t = r["recon_frames"][t].transpose(1, 2, 0)   # HWC
            psnr_per_frame.append(psnr(orig_t, recon_t, data_range=1.0))
            ssim_per_frame.append(ssim(orig_t, recon_t, data_range=1.0, channel_axis=2))

        avg_psnr = np.mean(psnr_per_frame)
        avg_ssim = np.mean(ssim_per_frame)
        psnr_vals.append(avg_psnr)
        ssim_vals.append(avg_ssim)
        l1_vals.append(r["loss"])

        n_sel = r["n_selected"]
        print(f"  {ratio:>6.2f} | {n_sel:>5}/{T*NUM_TOKENS} ({100*n_sel/(T*NUM_TOKENS):4.0f}%) | "
              f"{avg_psnr:>10.2f} | {avg_ssim:>8.4f} | {r['loss']:>10.4f}")

    print()
    print("참고: PSNR > 30 dB = 좋음, > 40 dB = 매우 좋음")
    print("     SSIM > 0.9 = 원본과 유사, > 0.95 = 매우 유사")

In [ ]:
if mae_model is not None and psnr_vals:
    ratios_valid = [r for r in RATIOS_TO_TEST if ratio_results[r]["recon_frames"] is not None]
    patch_counts = [ratio_results[r]["n_selected"] for r in ratios_valid]
    total_patches = T * NUM_TOKENS
    patch_pcts    = [100 * n / total_patches for n in patch_counts]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(patch_pcts, psnr_vals, "o-", color="steelblue", linewidth=2, markersize=8)
    axes[0].axhline(30, color="tomato", linestyle="--", alpha=0.5, label="30 dB (좋음)")
    axes[0].axhline(40, color="seagreen", linestyle="--", alpha=0.5, label="40 dB (매우 좋음)")
    for x, y, r in zip(patch_pcts, psnr_vals, ratios_valid):
        axes[0].annotate(f"ratio={r}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
    axes[0].set_xlabel("선택 패치 비율 (%)")
    axes[0].set_ylabel("PSNR (dB)")
    axes[0].set_title("패치 비율 vs PSNR")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(patch_pcts, ssim_vals, "o-", color="darkorange", linewidth=2, markersize=8)
    axes[1].axhline(0.9, color="tomato", linestyle="--", alpha=0.5, label="SSIM 0.9")
    axes[1].axhline(0.95, color="seagreen", linestyle="--", alpha=0.5, label="SSIM 0.95")
    for x, y, r in zip(patch_pcts, ssim_vals, ratios_valid):
        axes[1].annotate(f"ratio={r}", (x, y), textcoords="offset points", xytext=(5, -12), fontsize=8)
    axes[1].set_xlabel("선택 패치 비율 (%)")
    axes[1].set_ylabel("SSIM")
    axes[1].set_title("패치 비율 vs SSIM")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("AutoGaze 패치 비율에 따른 VideoMAE 복원 품질", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "quality_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"저장: {OUTPUT_DIR}/quality_curve.png")

---
## 8. 복원 영상 저장 (MP4)

In [ ]:
if mae_model is None or recon_frames is None:
    print("VideoMAE 없음 — 영상 저장 건너뜁니다.")
else:
    import imageio.v3 as iio

    def save_comparison_video(orig_vis, recon_vis, masked_mask, output_path, fps=4.0):
        """
        원본 | 선택 패치 | 복원 세 화면을 나란히 붙인 MP4 저장.
        orig_vis, recon_vis : (T, C, H, W) float32 [0,1]
        masked_mask         : gazing_mask[-1][0]  (T, N)
        """
        frames_out = []
        for t in range(len(orig_vis)):
            # 원본
            orig_hw   = (orig_vis[t].transpose(1, 2, 0) * 255).astype(np.uint8)
            # 선택 패치
            masked_hw = make_masked_view(orig_vis[t], masked_mask[t].cpu())
            # 복원
            recon_hw  = (recon_vis[t].transpose(1, 2, 0) * 255).astype(np.uint8)

            # 세 패널 가로로 합치기
            h = orig_hw.shape[0]
            # 레이블 추가
            combined = np.concatenate([orig_hw, masked_hw, recon_hw], axis=1)
            frames_out.append(combined)

        iio.imwrite(
            str(output_path),
            frames_out,
            fps=fps,
            codec="libx264",
            quality=8,
            macro_block_size=8,
        )
        return output_path

    largest_mask_vis = gaze_outputs["gazing_mask"][-1][0]

    # gazing_ratio=0.75 복원
    vid_path = OUTPUT_DIR / f"reconstruction_ratio{int(GAZING_RATIO*100):03d}.mp4"
    save_comparison_video(video_vis, recon_frames, largest_mask_vis, vid_path, fps=4.0)
    print(f"저장: {vid_path}")

    # gazing_ratio별 복원 영상
    for ratio, r in ratio_results.items():
        if r["recon_frames"] is None:
            continue
        lm = r["gaze_outputs"]["gazing_mask"][-1][0]
        vp = OUTPUT_DIR / f"reconstruction_ratio{int(ratio*100):03d}.mp4"
        save_comparison_video(video_vis, r["recon_frames"], lm, vp, fps=4.0)
        print(f"  ratio={ratio}: {vp.name}")

    print(f"\n모든 영상 저장 완료: {OUTPUT_DIR}/")

---
## 9. 핵심 정리

### 복원이 가능한 이유 — 3줄 요약

1. **VideoMAE는 MAE 기반**: 일부 패치만 보고 나머지를 복원하도록 사전 학습된 모델
2. **AutoGaze = VideoMAE의 'greedy oracle' 근사**: AutoGaze는 VideoMAE reconstruction loss를 reward로 학습 → 선택된 패치들이 VideoMAE 복원에 최적화된 패치들
3. **디코더가 mask_token으로 빈 자리 채움**: 선택 패치의 특징(features) + 위치(pos embedding) 정보로 나머지 패치 픽셀 예측

### 복원 품질의 한계

- 복원 결과는 VideoMAE의 표현력에 의존 (decoder 크기, 학습 데이터)
- 선택 패치가 적을수록 context 부족 → 세부 텍스처 손실
- 첫 프레임에 패치가 집중되는 경향 (Dirichlet 분포 alpha 설정 때문)
- **MLLM 활용 관점**: 복원 품질 자체가 목적이 아니라,  
  선택된 패치가 '정보의 대부분을 담고 있음'을 검증하는 지표

### 논문상 언급

> AutoGaze 논문 (CVPR 2026): VideoMAE reconstruction loss를 reward로 사용하므로,  
> 선택된 패치가 충분한 정보를 담으면 VideoMAE decoder가 전체를 복원할 수 있다.  
> `task_loss_requirement` 임계값은 이 복원 품질의 허용 하한선을 의미함.

### 다음 단계
- `02_train_ntp_ko.ipynb` — NTP 학습으로 더 좋은 패치 선택기 만들기
- `03_train_rl_ko.ipynb` — RL로 복원 품질 보상 최적화
- `04_validate_pseudolabels_ko.ipynb` — 선택 패치의 복원 품질로 pseudo-label 검증

---
## 10. (심화) NVILA — 복원 가능한 패치로 비디오 질의응답

VideoMAE 복원이 가능한 패치는 **언어 이해에도 충분한 시각 정보**를 담고 있습니다.  
**NVILA-8B-HD-Video**는 AutoGaze를 내장해 이 패치들로 바로 질의응답을 수행합니다.

```
복원 가능한 패치 선택
   (AutoGaze + VideoMAE RL 학습)
            │
            ▼
   [SigLIP/ViT]  →  vision_tokens
            │
            ▼
   [Qwen2-7B]  →  질의응답 생성
```

**연결 관계**:
- `task_loss_requirement` 임계값 → "VideoMAE가 N% 이상 복원 가능한 최소 패치"
- 이 패치들 = MLLM이 비디오를 이해하기에도 충분한 정보 포함
- NVILA는 AutoGaze를 내장 → 별도 VideoMAE 없이 end-to-end 추론 가능

**아키텍처 메모**:
- 텍스트 백본: Qwen2-7B (hidden_size=3584, 28 layers, 4 KV heads)
- 비전: SigLIP scales 56+112+196+392
- 비디오 토큰 플레이스홀더: `<vila/video>` (프롬프트에 명시 필요)
- `num_video_frames`는 AutoGaze `max_num_frames`(16)의 배수여야 함

> **사전 요건**: `bash scripts/download_models.sh weights nvila` (~16 GB)

In [ ]:
# ── NVILA로 비디오 질의응답 (AutoGaze 내장) ────────────────────────
NVILA_PATH = "../weights/NVILA-8B-HD-Video"

def _to_device_nvila(v, device):
    """BatchFeature 내 중첩 구조를 재귀적으로 device로 이동."""
    if isinstance(v, torch.Tensor):
        return v.to(device)
    if isinstance(v, list):
        return [_to_device_nvila(x, device) for x in v]
    if isinstance(v, dict):
        return {k: _to_device_nvila(vv, device) for k, vv in v.items()}
    return v

try:
    from transformers import AutoProcessor, AutoModel

    print(f"NVILA 로드 중: {NVILA_PATH}")

    nvila_processor = AutoProcessor.from_pretrained(
        NVILA_PATH,
        trust_remote_code=True,
        autogaze_model_id=NVILA_PATH,   # 로컬 AutoGaze 가중치 사용
    )
    # AutoGaze max_num_frames(16)의 배수 요건 충족
    nvila_processor.num_video_frames = 16

    nvila_model_r = AutoModel.from_pretrained(
        NVILA_PATH,
        trust_remote_code=True,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    nvila_model_r.eval()
    print("NVILA 로드 완료 ✓")

    # ── AutoGaze 단독 실행 결과 요약 ──────────────────────────────
    print(f"\nAutoGaze 패치 선택 현황 (gazing_ratio={GAZING_RATIO}):")
    print(f"  선택: {n_real} / {T * NUM_TOKENS} 패치 ({100*n_real/(T*NUM_TOKENS):.0f}%)")
    if recon_loss is not None:
        print(f"  VideoMAE 복원 L1: {recon_loss:.4f}  → 이 패치로 영상 복원 가능")
    print(f"  → 복원이 가능한 패치 = MLLM 이해에도 충분한 시각 정보")

    # ── NVILA 추론 ─────────────────────────────────────────────────
    # <vila/video> 토큰을 프롬프트에 명시해야 processor가 video token을 올바르게 확장
    video_token = nvila_processor.tokenizer.video_token   # '<vila/video>'
    question    = "이 비디오에서 무엇이 일어나고 있나요?"
    prompt      = f"{video_token}\n{question}"

    _nvila_device = next(nvila_model_r.parameters()).device
    inputs = nvila_processor(text=prompt, videos=str(Path(VIDEO_PATH).resolve()))
    inputs_dev = _to_device_nvila(dict(inputs), _nvila_device)
    for key in ("input_ids", "attention_mask"):
        if key in inputs_dev and isinstance(inputs_dev[key], list):
            inputs_dev[key] = torch.tensor(inputs_dev[key], device=_nvila_device)
    input_ids  = inputs_dev.pop("input_ids")
    extra_kw   = inputs_dev

    print(f"\n질문: {question}")
    print("NVILA 추론 중 ...")

    with torch.inference_mode():
        gen_ids = nvila_model_r.generate(
            input_ids=input_ids,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
            top_p=None,
            **extra_kw,
        )

    answer = nvila_processor.batch_decode(
        gen_ids[:, input_ids.shape[1]:], skip_special_tokens=True,
    )[0].strip()
    print(f"\n[NVILA 답변]\n{answer}")

    print(f"\n{'─'*55}")
    print("AutoGaze + VideoMAE  ↔  NVILA 역할 비교:")
    print("  AutoGaze + VideoMAE : 복원 가능성으로 패치 품질 검증 (학습/평가용)")
    print("  NVILA               : 선택된 패치로 언어 질의응답 직접 수행 (배포용)")
    print("  공통점              : 동일한 AutoGaze 가중치, 동일한 패치 선택 기준")

except FileNotFoundError:
    print(f"NVILA 가중치 없음: {NVILA_PATH}")
    print("  bash scripts/download_models.sh weights nvila  (~16 GB)")
except Exception as e:
    print(f"NVILA 실행 실패: {type(e).__name__}: {e}")